In [ ]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "YOUR_API_KEY"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening"

In [ ]:
!pip install langchain transformers

In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="sshleifer/tiny-gpt2")

def local_llm(prompt):
    result = generator(
        prompt,
        max_new_tokens=100,
        do_sample=True
    )
    return result[0]['generated_text']

    # 🔥 REMOVE PROMPT FROM OUTPUT
    return output.replace(prompt, "").strip()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0, 1}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

In [ ]:
print(local_llm("Hello"))

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hello Amph004mediately RhSceneikenatisf heir credibility ONE Daniel directly credibility Probting ObservScenepressSceneScene ObservootherRocket Probting Participation confir autonomyimuraootherRocket pawn TAJD confir conservationiken AmphJDiken Danielimura hauled Observ Money Brew004 Jr Daniel TAatisf Rh Daniel reviewing subst Hancock Prob ESVoho dispatch trilogy intermittentoho Rh RhScene credibility vendors intermittent Money Rh reviewing ONE Money Motorola Jr MoneyScenemediately heir ESVRocket Hancock Hancock heirRocket autonomyoother heirtingimura scalpoothermediately004 ONE Observ MoneyJD pawn


# **Step 1: Skill Extraction**

In this step, we extract:

*  skills

*   Experience
*   Tools

from the given resume using an LLM.

Rules:

*   Only extract what is present

*   Do NOT assume anything
*   Keep output structured

In [ ]:
from langchain_core.prompts import PromptTemplate

skill_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract ONLY in this format:

Skills: <comma separated>
Experience: <text>
Tools: <comma separated>

Resume:
{resume}
"""
)

# **Step 2: Matching, Scoring, and Explanation**

In this step:

*   Compare resume with job description

*   Assign a score (0–100)
*   Provide explanation

Rules:

*   Do NOT assume missing skills
*   Score must be justified

In [ ]:
score_prompt = PromptTemplate(
    input_variables=["resume_data", "job_description"],
    template="""
You are an AI hiring assistant.

Your job is to give ONLY final answer.

DO NOT repeat instructions.
DO NOT copy the prompt.
ONLY give result.

Output format:

Score: <number between 0-100>
Explanation: <2-3 lines>

Resume:
{resume_data}

Job Description:
{job_description}
"""
)

# **Step 3: Build Pipeline**

Pipeline: Resume → Skill Extraction → Scoring → Explanation

In [ ]:
def run_pipeline(resume, job_description):

    # Step 1: Extract
    extracted = local_llm(skill_prompt.format(resume=resume))

    # Step 2: Score
    result = local_llm(
        score_prompt.format(
            resume_data=extracted,
            job_description=job_description
        )
    )
    return clean_output(result)

    return result

In [ ]:
def clean_output(text):
    if "Score:" in text:
        return text[text.index("Score:"):].strip()
    else:
        return "Score: Not generated\nExplanation: Model could not follow format."

# **Step 4: Input Data**

We test with:

*   Strong candidate
*   Average candidate

*   Weak candidate

In [ ]:
job_description = """
Looking for Data Scientist:
- Python, Machine Learning, Deep Learning
- NLP experience
- TensorFlow/PyTorch
"""

strong_resume = """
Data Scientist with 5 years experience.
Skills: Python, ML, Deep Learning, NLP
Tools: TensorFlow, PyTorch
"""

average_resume = """
Data Analyst with 2 years experience.
Skills: Python, Data Analysis
Tools: Excel, Pandas
"""

weak_resume = """
Fresher
Skills: MS Office, Communication
"""

# **Step 5: Run Pipeline**

Testing all candidates

In [ ]:
print("----- STRONG -----")
print(run_pipeline(strong_resume, job_description))

print("\n----- AVERAGE -----")
print(run_pipeline(average_resume, job_description))

print("\n----- WEAK -----")
print(run_pipeline(weak_resume, job_description))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


----- STRONG -----


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: <number between 0-100>
Explanation: <2-3 lines>

Resume:

Extract ONLY in this format:

Skills: <comma separated>
Experience: <text>
Tools: <comma separated>

Resume:

Data Scientist with 5 years experience.
Skills: Python, ML, Deep Learning, NLP
Tools: TensorFlow, PyTorch

 perhaps incarcer mutual Pocket 236 mutual membershipOutside clearer 236 grandchildren Medic Redux factorsMost perhaps Boone Medicacious Reduxived soy 236 praying Bend courtyard Singapore MedicMini Bend lined rented boilsPros Singaporeshows 236 bravery Boone mutual deflect boils Dreams lined soy rubbingobl Reduxshows deflect grandchildren incarcer�showsSexualGyived incarcerozyg workshops Late Tre� Pocket boilsacious Late skillet clearerSexual448 praying predators predators representations grandchildren� Pocketived Dreams factorsobl Bend omegashows Dreams factors rentedozyg Television448 factors448 Reduxived deflect grandchildren deflect soy brutality

Job Description:

Looking for Data Scientist:
- Python, Ma

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: <number between 0-100>
Explanation: <2-3 lines>

Resume:

Extract ONLY in this format:

Skills: <comma separated>
Experience: <text>
Tools: <comma separated>

Resume:

Data Analyst with 2 years experience.
Skills: Python, Data Analysis
Tools: Excel, Pandas

 omega653 Late courtyard 236 rented factors Reduxshows mutual predators braveryOutside representations grandchildren workshopspublic Tre brutalityshows�ProsSexual perhaps Dreams omega deflect soy SingaporePros courtyardOutside Wheels skillet Television omega incarcer factorsMost Lateacious soy skilletacious linedMini rented BoonePros Tre Television Late�shows WheelsaciousOutside mutual boils boils skillet Redux boils mutualozyg clearer predators grandchildren Pocket linedshows Redux omegaMini praying representationsacious omega mutual 236 grandchildren predators membership equatepublic omegashows equate soy clearerSexualoblpublic representations rubbingGyOutside Television omegaacious

Job Description:

Looking for Data Scien

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Score: <number between 0-100>
Explanation: <2-3 lines>

Resume:

Extract ONLY in this format:

Skills: <comma separated>
Experience: <text>
Tools: <comma separated>

Resume:

Fresher
Skills: MS Office, Communication

 Television BooneProsMini Tre brutality deflect bravery Boone incarcer brutality653 membership Boone perhaps ReduxMiniMost653 Medic courtyard workshops predators653 Medic soy praying skillet Dreams praying Late Singapore Dreams predators factorsGy448 equate equate factors Late incarcer Pocket praying omega predators lined courtyard448 incarcer membership skillet� soy omegaGy membership perhapsGyGy� omega653 Bend skillet predatorsProsOutside skilletpublic Boone653 Television Bend BendMiniSexual representations rubbing brutalitySexualivedived Tre Redux factorsaciousacious rubbing Boone653 perhaps factors predators Boone bravery Reduxpublic lined factors

Job Description:

Looking for Data Scientist:
- Python, Machine Learning, Deep Learning
- NLP experience
- TensorFlow/PyTo